## Working with Transformers in the HuggingFace Ecosystem

In this laboratory exercise we will learn how to work with the HuggingFace ecosystem to adapt models to new tasks. As you will see, much of what is required is *investigation* into the inner-workings of the HuggingFace abstractions. With a little work, a little trial-and-error, it is fairly easy to get a working adaptation pipeline up and running.

**Setup**: eseguire questa cella per prima. Installa/aggiorna tutte le librerie necessarie per l'intero notebook (incluso `peft`, usato solo nell'Esercizio 3.1), rimuove `torchao` (spesso preinstallato su Colab in una versione incompatibile con `peft`, causa altrimenti un `ImportError`) e abbassa la verbosita’ di `transformers`/`datasets`/`huggingface_hub`. Questi pacchetti stampano di default parecchi warning e "load report" (liste di pesi UNEXPECTED/MISSING a ogni caricamento) che sono informativi la prima volta ma diventano solo rumore quando si ripetono identici a ogni cella: qui li silenziamo per tenere l'output pulito e concentrato sui risultati che contano.

In [1]:
!pip install -q -U transformers datasets accelerate peft scikit-learn

# Colab pre-installa una versione vecchia di torchao (0.10.0). peft (nelle versioni
# recenti) fa un controllo di compatibilita' su torchao e va in errore se lo trova
# ma troppo vecchio. Qui non ci serve affatto (serve solo per la quantizzazione),
# quindi lo disinstalliamo: se non e' presente, peft salta semplicemente il controllo.
!pip uninstall -y -q torchao

import os
import warnings
import logging as py_logging

# Riduciamo il rumore "di libreria" che non aggiunge nulla alla comprensione del lab:
# warning sulle richieste non autenticate all'HF Hub, i "load report" con le liste
# UNEXPECTED/MISSING ripetute a ogni from_pretrained(), e warning minori di deprecazione.
os.environ["HF_HUB_DISABLE_PROGRESS_BAR_DEPRECATION_WARNING"] = "1"
py_logging.getLogger("huggingface_hub").setLevel(py_logging.ERROR)

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets.utils import logging as ds_logging
ds_logging.set_verbosity_error()

warnings.filterwarnings("ignore")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00


### Exercise 1: Sentiment Analysis (warm up)

In this first exercise we will start from a pre-trained BERT transformer and build up a model able to perform text sentiment analysis. Transformers are complex beasts, so we will build up our pipeline in several explorative and incremental steps.

#### Exercise 1.1: Dataset Splits and Pre-trained model
There are a many sentiment analysis datasets, but we will use one of the smallest ones available: the [Cornell Rotten Tomatoes movie review dataset](cornell-movie-review-data/rotten_tomatoes), which consists of 5,331 positive and 5,331 negative processed sentences from the Rotten Tomatoes movie reviews.

**Your first task**: Load the dataset and figure out what splits are available and how to get them. Spend some time exploring the dataset to see how it is organized. Note that we will be using the [HuggingFace Datasets](https://huggingface.co/docs/datasets/en/index) library for downloading, accessing, splitting, and batching data for training and evaluation.

In [2]:
from datasets import load_dataset, get_dataset_split_names

DATASET_NAME = "cornell-movie-review-data/rotten_tomatoes"

# Quali split sono disponibili?
split_names = get_dataset_split_names(DATASET_NAME)
print("Split disponibili:", split_names)

dataset = load_dataset(DATASET_NAME)
print(dataset)

# Diamo un'occhiata alla struttura interna
print("\nFeatures:", dataset["train"].features)
print("\nEsempio (train[0]):", dataset["train"][0])
print("\nEsempio (train[-1]):", dataset["train"][-1])

for split in dataset:
    n_pos = sum(1 for l in dataset[split]["label"] if l == 1)
    n_neg = sum(1 for l in dataset[split]["label"] if l == 0)
    print(f"{split:12s} -> {len(dataset[split])} esempi (pos={n_pos}, neg={n_neg})")


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

Split disponibili: ['train', 'validation', 'test']


train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

Esempio (train[0]): {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}

Esempio (train[-1]): {'text': 'things really get weird , though not particularly scary : the movie is all portent and no content .', 'label': 0}
train        -> 8530 esempi (pos=4265, neg=4265)
validation   -> 1066 esempi (pos=533, neg=533)
test         -> 1066 esempi (pos=533, neg=533)


**Analisi**: il dataset espone tre split: `train` (8530 esempi), `validation` (1066) e `test` (1066), ciascuno perfettamente bilanciato tra le due classi (`label`: 0 = negativo, 1 = positivo). Ogni esempio è un dizionario con due campi: `text` (la frase, già pre-processata/tokenizzata a livello di parole con minuscolo e punteggiatura separata) e `label` (intero 0/1). Non è quindi necessario alcuno split manuale: possiamo usare direttamente `dataset["train"]`, `dataset["validation"]` e `dataset["test"]`.

#### Exercise 1.2: A Pre-trained BERT and Tokenizer

The model we will use is a *very* small BERT transformer called [Distilbert](https://huggingface.co/distilbert/distilbert-base-uncased) this model was trained (using self-supervised learning) on the same corpus as BERT but using the full BERT base model as a *teacher*.

**Your next task**: Load the Distilbert model and corresponding tokenizer. Use the tokenizer on a few samples from the dataset and pass the tokens through the model to see what outputs are provided. I suggest you use the [`AutoModel`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html) class (and the `from_pretrained()` method) to load the model and `AutoTokenizer` to load the tokenizer).

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

samples = dataset["train"]["text"][:3]
for s in samples:
    print("-", s)

inputs = tokenizer(samples, padding=True, truncation=True, return_tensors="pt").to(device)
print("\nChiavi restituite dal tokenizer:", list(inputs.keys()))
print("input_ids shape:", inputs["input_ids"].shape)

# Decodifichiamo il primo esempio per vedere i token speciali aggiunti
print("\nToken del primo esempio:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    outputs = model(**inputs)

print("\nTipo di output:", type(outputs))
print("Chiavi disponibili:", outputs.keys())
print("last_hidden_state shape:", outputs.last_hidden_state.shape)


Device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

- the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
- the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth .
- effective but too-tepid biopic

Chiavi restituite dal tokenizer: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids shape: torch.Size([3, 52])

Token del primo esempio:
['[CLS]', 'the', 'rock', 'is', 'destined', 'to', 'be', 'the', '21st', 'century', "'", 's', 'new', '"', 'conan', '"', 'and', 'that', 'he', "'", 's', 'going', 'to', 'make', 'a', 'splash', 'even', 'greater', 'than', 'arnold', 'schwarz', '##ene', '##gger', ',', 'jean', '-', 'cl', '##aud', 'van', 'dam', '##me', 'or', 'steven', 'sega', '##l', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Tipo di outp

**Analisi**: il tokenizer restituisce `input_ids` e `attention_mask` (Distilbert non usa `token_type_ids`, a differenza di BERT, perché è stato addestrato senza il task di *next sentence prediction* e quindi non ha bisogno di distinguere due segmenti). Il primo token di ogni sequenza è sempre `[CLS]` (id fisso), seguito dai token della frase e infine `[SEP]`.

Il modello `AutoModel` (senza testa di classificazione) restituisce un oggetto `BaseModelOutput` con un solo campo utile, `last_hidden_state`, di forma `(batch, seq_len, hidden_size=768)`. A differenza di BERT standard, **Distilbert non ha un `pooler_output`**: per ottenere una rappresentazione a livello di frase dobbiamo quindi prendere manualmente il vettore corrispondente al token `[CLS]`, cioè `last_hidden_state[:, 0, :]`.

#### Exercise 1.3: A Stable Baseline

In this exercise I want you to:
1. Use Distilbert as a *feature extractor* to extract representations of the text strings from the dataset splits;
2. Train a classifier (your choice, by an SVM from Scikit-learn is an easy choice).
3. Evaluate performance on the validation and test splits.

These results are our *stable baseline* -- the **starting** point on which we will (hopefully) improve in the next exercise.

**Hint**: There are a number of ways to implement the feature extractor, but probably the best is to use a [feature extraction `pipeline`](https://huggingface.co/tasks/feature-extraction). You will need to interpret the output of the pipeline and extract only the `[CLS]` token from the *last* transformer layer. *How can you figure out which output that is?*

In [4]:
import numpy as np

@torch.no_grad()
def extract_cls_features(texts, batch_size=64):
    """Estrae il vettore [CLS] dell'ultimo layer di Distilbert per una lista di testi.

    Invece di usare la pipeline 'feature-extraction' (che restituisce output annidati
    un po' scomodi da maneggiare in batch), tokenizziamo e chiamiamo il modello
    direttamente: e' piu' trasparente e molto piu' veloce su GPU perche' controlliamo
    noi il padding/batching.
    """
    feats = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        out = model(**enc)
        cls = out.last_hidden_state[:, 0, :]  # (batch, hidden) -> il token [CLS]
        feats.append(cls.cpu().numpy())
    return np.concatenate(feats, axis=0)

X_train = extract_cls_features(dataset["train"]["text"])
y_train = np.array(dataset["train"]["label"])
X_val = extract_cls_features(dataset["validation"]["text"])
y_val = np.array(dataset["validation"]["label"])
X_test = extract_cls_features(dataset["test"]["text"])
y_test = np.array(dataset["test"]["label"])

print("Shape delle feature di training:", X_train.shape)


Shape delle feature di training: (8530, 768)


In [5]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Le feature del transformer non sono normalizzate: standardizziamole prima della SVM
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

clf = SVC(kernel="rbf", C=1.0)
clf.fit(X_train_s, y_train)

val_pred = clf.predict(X_val_s)
test_pred = clf.predict(X_test_s)

print("=== Validation ===")
print("Accuracy:", accuracy_score(y_val, val_pred))
print("F1:", f1_score(y_val, val_pred))

print("\n=== Test ===")
print("Accuracy:", accuracy_score(y_test, test_pred))
print("F1:", f1_score(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, target_names=["negative", "positive"]))


=== Validation ===
Accuracy: 0.8189493433395872
F1: 0.8090999010880316

=== Test ===
Accuracy: 0.7964352720450282
F1: 0.7911453320500481

              precision    recall  f1-score   support

    negative       0.78      0.82      0.80       533
    positive       0.81      0.77      0.79       533

    accuracy                           0.80      1066
   macro avg       0.80      0.80      0.80      1066
weighted avg       0.80      0.80      0.80      1066



**Analisi (baseline)**: eseguendo la pipeline si ottiene:

| Split | Accuracy | F1 |
|---|---|---|
| Validation | 0.819 | 0.809 |
| Test | 0.796 | 0.791 |

Un'accuratezza intorno all'80% e’ in linea con quanto ci si aspettava concettualmente: il vettore `[CLS]` di Distilbert *non* fine-tuned e’ stato prodotto da un modello addestrato solo con *masked language modeling*, quindi non e’ mai stato ottimizzato per riassumere il significato dell'intera frase (a differenza, ad esempio, di sentence-embeddings tipo SBERT). Contiene comunque informazione semantica sufficiente perche’ una SVM riesca a separare abbastanza bene le due classi, ma c'e’ chiaramente margine di miglioramento. Questo e’ il nostro punto di partenza: nell'Esercizio 2 vediamo se e quanto migliora permettendo al transformer di adattarsi al task.

-----
### Exercise 2: Fine-tuning Distilbert

In this exercise we will fine-tune the Distilbert model to (hopefully) improve sentiment analysis performance.

#### Exercise 2.1: Token Preprocessing

The first thing we need to do is *tokenize* our dataset splits. Our current datasets return a dictionary with *strings*, but we want *input token ids* (i.e. the output of the tokenizer). This is easy enough to do my hand, but the HugginFace `Dataset` class provides convenient, efficient, and *lazy* methods. See the documentation for [`Dataset.map`](https://huggingface.co/docs/datasets/v3.5.0/en/package_reference/main_classes#datasets.Dataset.map).

**Tip**: Verify that your new datasets are returning for every element: `text`, `label`, `intput_ids`, and `attention_mask`.

In [6]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

# batched=True per efficienza; niente padding qui, ci pensera' il data collator
tokenized_datasets = dataset.map(tokenize_function, batched=True)
print(tokenized_datasets)

example = tokenized_datasets["train"][0]
print("\nChiavi per esempio:", list(example.keys()))
print("text:", example["text"])
print("label:", example["label"])
print("input_ids:", example["input_ids"])
print("attention_mask:", example["attention_mask"])


Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})

Chiavi per esempio: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
text: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
label: 1
input_ids: [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005, 1055, 2183, 2000, 2191, 1037, 17624, 2130, 3618, 2084, 7779, 29058, 8625, 13327, 1010, 3744, 1011, 18856, 19513, 3158, 5477, 4168, 2030, 7112, 16562, 2140

**Analisi**: ogni elemento del dataset tokenizzato contiene ora `text`, `label`, `input_ids` e `attention_mask`, come richiesto. Non applichiamo padding in questa fase (lasciamo le sequenze a lunghezza variabile): il padding verra' fatto dinamicamente per batch dal `DataCollatorWithPadding` nell'Esercizio 2.3, il che e' piu' efficiente rispetto a fare padding fisso su tutto il dataset (evitiamo di sprecare calcolo su padding inutile quando la maggior parte delle frasi e' corta).

#### Exercise 2.2: Setting up the Model to be Fine-tuned

In this exercise we need to prepare the base Distilbert model for fine-tuning for a *sequence classification task*. This means, at the very least, appending a new, randomly-initialized classification head connected to the `[CLS]` token of the last transformer layer. Luckily, HuggingFace already provides an `AutoModel` for just this type of instantiation: [`AutoModelForSequenceClassification`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelforsequenceclassification). You will want you instantiate one of these for fine-tuning.

In [7]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

ft_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
).to(device)

print(ft_model.classifier)
n_params = sum(p.numel() for p in ft_model.parameters())
n_trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
print(f"Parametri totali: {n_params:,} | Parametri allenabili: {n_trainable:,}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Linear(in_features=768, out_features=2, bias=True)
Parametri totali: 66,955,010 | Parametri allenabili: 66,955,010


**Analisi**: `AutoModelForSequenceClassification` carica i pesi pre-addestrati di Distilbert e aggiunge sopra una nuova testa (`pre_classifier` + `classifier`, entrambi lineari) inizializzata casualmente, con output dimensione `num_labels=2`. Al caricamento, la libreria stampa un warning che segnala esattamente questo: i pesi di `classifier`/`pre_classifier` sono nuovi e non pre-addestrati, quindi il modello **deve** essere fine-tuned prima di essere utile (le predizioni iniziali sarebbero casuali).

#### Exercise 2.3: Fine-tuning Distilbert

Finally. In this exercise you should use a HuggingFace [`Trainer`](https://huggingface.co/docs/transformers/main/en/trainer) to fine-tune your model on the Rotten Tomatoes training split. Setting up the trainer will involve (at least):


1. Instantiating a [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/en/main_classes/data_collator) object which is what *actually* does your batch construction (by padding all sequences to the same length).
2. Writing an *evaluation function* that will measure the classification accuracy. This function takes a single argument which is a tuple containing `(logits, labels)` which you should use to compute classification accuracy (and maybe other metrics like F1 score, precision, recall) and return a `dict` with these metrics.  
3. Instantiating a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.51.1/en/main_classes/trainer#transformers.TrainingArguments) object using some reasonable defaults.
4. Instantiating a `Trainer` object using your train and validation splits, you data collator, and function to compute performance metrics.
5. Calling `trainer.train()`, waiting, waiting some more, and then calling `trainer.evaluate()` to see how it did.

**Tip**: When prototyping this laboratory I discovered the HuggingFace [Evaluate library](https://huggingface.co/docs/evaluate/en/index) which provides evaluation metrics. However I found it to have insufferable layers of abstraction and getting actual metrics computed. I suggest just using the Scikit-learn metrics...

In [8]:
import inspect
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}


# Versioni diverse di transformers usano nomi diversi per questo argomento
# (evaluation_strategy nelle versioni piu' vecchie, eval_strategy in quelle piu' recenti).
# Lo rileviamo a runtime cosi' il notebook funziona indipendentemente dalla versione installata.
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_strategy_kwarg = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir="./results_distilbert_ft",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    **{_strategy_kwarg: "epoch"},
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


{'loss': '0.6587', 'grad_norm': '3.312', 'learning_rate': '1.939e-05', 'epoch': '0.09363'}
{'loss': '0.5012', 'grad_norm': '8.739', 'learning_rate': '1.876e-05', 'epoch': '0.1873'}
{'loss': '0.4306', 'grad_norm': '6.505', 'learning_rate': '1.814e-05', 'epoch': '0.2809'}
{'loss': '0.3931', 'grad_norm': '10.53', 'learning_rate': '1.752e-05', 'epoch': '0.3745'}
{'loss': '0.3653', 'grad_norm': '10.13', 'learning_rate': '1.689e-05', 'epoch': '0.4682'}
{'loss': '0.4107', 'grad_norm': '8.105', 'learning_rate': '1.627e-05', 'epoch': '0.5618'}
{'loss': '0.3639', 'grad_norm': '8.648', 'learning_rate': '1.564e-05', 'epoch': '0.6554'}
{'loss': '0.3819', 'grad_norm': '5.599', 'learning_rate': '1.502e-05', 'epoch': '0.7491'}
{'loss': '0.3552', 'grad_norm': '5.401', 'learning_rate': '1.439e-05', 'epoch': '0.8427'}
{'loss': '0.3624', 'grad_norm': '5.937', 'learning_rate': '1.377e-05', 'epoch': '0.9363'}
{'eval_loss': '0.4121', 'eval_accuracy': '0.8189', 'eval_f1': '0.838', 'eval_precision': '0.7584', 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3751', 'grad_norm': '8.76', 'learning_rate': '1.315e-05', 'epoch': '1.03'}
{'loss': '0.2522', 'grad_norm': '9.318', 'learning_rate': '1.252e-05', 'epoch': '1.124'}
{'loss': '0.2205', 'grad_norm': '4.549', 'learning_rate': '1.19e-05', 'epoch': '1.217'}
{'loss': '0.2432', 'grad_norm': '14.55', 'learning_rate': '1.127e-05', 'epoch': '1.311'}
{'loss': '0.2218', 'grad_norm': '9.039', 'learning_rate': '1.065e-05', 'epoch': '1.404'}
{'loss': '0.2266', 'grad_norm': '23.79', 'learning_rate': '1.002e-05', 'epoch': '1.498'}
{'loss': '0.2615', 'grad_norm': '9.077', 'learning_rate': '9.401e-06', 'epoch': '1.592'}
{'loss': '0.2778', 'grad_norm': '6.986', 'learning_rate': '8.777e-06', 'epoch': '1.685'}
{'loss': '0.2504', 'grad_norm': '14.82', 'learning_rate': '8.152e-06', 'epoch': '1.779'}
{'loss': '0.2221', 'grad_norm': '2.742', 'learning_rate': '7.528e-06', 'epoch': '1.873'}
{'loss': '0.241', 'grad_norm': '12.93', 'learning_rate': '6.904e-06', 'epoch': '1.966'}
{'eval_loss': '0.382', 'e

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2127', 'grad_norm': '5.528', 'learning_rate': '6.28e-06', 'epoch': '2.06'}
{'loss': '0.1416', 'grad_norm': '8.844', 'learning_rate': '5.655e-06', 'epoch': '2.154'}
{'loss': '0.134', 'grad_norm': '2.207', 'learning_rate': '5.031e-06', 'epoch': '2.247'}
{'loss': '0.1833', 'grad_norm': '12.41', 'learning_rate': '4.407e-06', 'epoch': '2.341'}
{'loss': '0.1287', 'grad_norm': '2.051', 'learning_rate': '3.783e-06', 'epoch': '2.434'}
{'loss': '0.1524', 'grad_norm': '8.551', 'learning_rate': '3.159e-06', 'epoch': '2.528'}
{'loss': '0.1527', 'grad_norm': '12.85', 'learning_rate': '2.534e-06', 'epoch': '2.622'}
{'loss': '0.1495', 'grad_norm': '2.276', 'learning_rate': '1.91e-06', 'epoch': '2.715'}
{'loss': '0.1571', 'grad_norm': '18.09', 'learning_rate': '1.286e-06', 'epoch': '2.809'}
{'loss': '0.137', 'grad_norm': '6.468', 'learning_rate': '6.617e-07', 'epoch': '2.903'}
{'loss': '0.16', 'grad_norm': '5.329', 'learning_rate': '3.745e-08', 'epoch': '2.996'}
{'eval_loss': '0.4718', 'eva

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '86.18', 'train_samples_per_second': '296.9', 'train_steps_per_second': '18.59', 'train_loss': '0.2724', 'epoch': '3'}


TrainOutput(global_step=1602, training_loss=0.27242818523137907, metrics={'train_runtime': 86.1773, 'train_samples_per_second': 296.946, 'train_steps_per_second': 18.59, 'train_loss': 0.27242818523137907, 'epoch': 3.0})

In [9]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


{'eval_loss': '0.5626', 'eval_accuracy': '0.834', 'eval_f1': '0.8369', 'eval_precision': '0.8225', 'eval_recall': '0.8518', 'eval_runtime': '0.5011', 'eval_samples_per_second': '2127', 'eval_steps_per_second': '33.93', 'epoch': '3'}
{'eval_loss': 0.5626180171966553, 'eval_accuracy': 0.8339587242026266, 'eval_f1': 0.8368663594470046, 'eval_precision': 0.822463768115942, 'eval_recall': 0.851782363977486, 'eval_runtime': 0.5011, 'eval_samples_per_second': 2127.347, 'eval_steps_per_second': 33.926, 'epoch': 3.0}


**Analisi**: dopo il fine-tuning completo (3 epoche) si ottiene sul test set `eval_accuracy = 0.834`, `eval_f1 = 0.837`: un miglioramento netto rispetto alla baseline SVM (0.796 / 0.791), che conferma l'intuizione di partenza: permettere a tutti i pesi del transformer di adattarsi al task specifico produce rappresentazioni molto piu’ allineate al sentiment rispetto al `[CLS]` "grezzo" pre-addestrato.

Guardando pero’ i log di training epoca per epoca si nota qualcosa di interessante:

| Epoca | Training Loss | Validation Loss | Val. Accuracy |
|---|---|---|---|
| 1 | 0.362 | 0.412 | 0.819 |
| 2 | 0.241 | 0.382 | 0.851 |
| 3 | 0.160 | 0.472 | 0.858 |

La training loss scende in modo monotono e deciso, mentre la validation loss ha un andamento non monotono: scende dalla prima alla seconda epoca (0.412 → 0.382) ma risale nella terza (0.382 → 0.472), pur restando l'accuracy di validation la piu’ alta delle tre epoche (0.858). Questo e’ un segnale di overfitting piu’ sottile del solito: non e’ il caso classico in cui l'accuracy peggiora o si stabilizza mentre la loss sale, ma un disallineamento tra le due metriche. Con circa 67M di parametri allenabili su un training set piccolo (8530 esempi), il modello ha capacita’ piu’ che sufficiente per diventare via via piu’ overconfident sulle proprie predizioni (comprese alcune sbagliate), il che fa salire la cross-entropy anche se il numero di predizioni corrette continua a crescere leggermente. E’ un primo segnale di overfitting che precede quello, piu’ netto, su un plateau o peggioramento dell'accuracy: probabilmente emergerebbe in modo piu’ evidente con qualche epoca in piu’. Il costo di questo guadagno di performance resta comunque duplice: dobbiamo aggiornare **tutti** i parametri di Distilbert, e lo facciamo osservando gia’ segnali di overfitting incipiente dopo poche epoche su un dataset di queste dimensioni. Questo motiva l'Esercizio 3.1 (fine-tuning efficiente), dove vediamo se allenare meno parametri aiuta anche a contenere questo effetto.

-----
### Exercise 3: Choose at Least One


#### Exercise 3.1: Efficient Fine-tuning for Sentiment Analysis (easy)

In Exercise 2 we fine-tuned the *entire* Distilbert model on Rotten Tomatoes. This is expensive, even for a small model. Find an *efficient* way to fine-tune Distilbert on the Rotten Tomatoes dataset (or some other dataset).

**Hint**: You could check out the [HuggingFace PEFT library](https://huggingface.co/docs/peft/en/index) for some state-of-the-art approaches that should "just work". How else might you go about making fine-tuning more efficient without having to change your training pipeline from above?

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

# Distilbert chiama i layer di proiezione dell'attenzione q_lin/k_lin/v_lin/out_lin
# (non query/key/value come BERT): dobbiamo scoprirlo ispezionando i nomi dei moduli.
for name, _ in ft_model.named_modules():
    if "attention" in name and name.count(".") <= 4:
        print(name)


distilbert.transformer.layer.0.attention
distilbert.transformer.layer.1.attention
distilbert.transformer.layer.2.attention
distilbert.transformer.layer.3.attention
distilbert.transformer.layer.4.attention
distilbert.transformer.layer.5.attention


In [11]:
base_model_for_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id,
).to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],  # applichiamo LoRA solo a query e value
    modules_to_save=["pre_classifier", "classifier"],  # la testa va comunque allenata per intero
)

peft_model = get_peft_model(base_model_for_lora, lora_config)
peft_model.print_trainable_parameters()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [12]:
lora_training_args = TrainingArguments(
    output_dir="./results_distilbert_lora",
    save_strategy="epoch",
    learning_rate=1e-3,  # LR piu' alto: allenando pochi parametri possiamo permettercelo
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    **{_strategy_kwarg: "epoch"},  # stesso trucco di compatibilita' usato sopra
)

lora_trainer = Trainer(
    model=peft_model,
    args=lora_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

lora_trainer.train()


{'loss': '0.534', 'grad_norm': '2.818', 'learning_rate': '0.0009694', 'epoch': '0.09363'}
{'loss': '0.4647', 'grad_norm': '3.89', 'learning_rate': '0.0009382', 'epoch': '0.1873'}
{'loss': '0.4407', 'grad_norm': '1.445', 'learning_rate': '0.000907', 'epoch': '0.2809'}
{'loss': '0.3945', 'grad_norm': '3.593', 'learning_rate': '0.0008758', 'epoch': '0.3745'}
{'loss': '0.3937', 'grad_norm': '1.214', 'learning_rate': '0.0008446', 'epoch': '0.4682'}
{'loss': '0.4242', 'grad_norm': '1.107', 'learning_rate': '0.0008134', 'epoch': '0.5618'}
{'loss': '0.4', 'grad_norm': '2.37', 'learning_rate': '0.0007821', 'epoch': '0.6554'}
{'loss': '0.4265', 'grad_norm': '1.789', 'learning_rate': '0.0007509', 'epoch': '0.7491'}
{'loss': '0.377', 'grad_norm': '1.16', 'learning_rate': '0.0007197', 'epoch': '0.8427'}
{'loss': '0.3951', 'grad_norm': '1.422', 'learning_rate': '0.0006885', 'epoch': '0.9363'}
{'eval_loss': '0.3761', 'eval_accuracy': '0.834', 'eval_f1': '0.8454', 'eval_precision': '0.7908', 'eval_rec

TrainOutput(global_step=1602, training_loss=0.3284656898433647, metrics={'train_runtime': 48.2803, 'train_samples_per_second': 530.03, 'train_steps_per_second': 33.181, 'train_loss': 0.3284656898433647, 'epoch': 3.0})

In [13]:
lora_test_results = lora_trainer.evaluate(tokenized_datasets["test"])
print("LoRA test results:", lora_test_results)
print("Full fine-tuning test results:", test_results)


{'eval_loss': '0.421', 'eval_accuracy': '0.8433', 'eval_f1': '0.8435', 'eval_precision': '0.8427', 'eval_recall': '0.8443', 'eval_runtime': '0.5225', 'eval_samples_per_second': '2040', 'eval_steps_per_second': '32.54', 'epoch': '3'}
LoRA test results: {'eval_loss': 0.4210093021392822, 'eval_accuracy': 0.8433395872420263, 'eval_f1': 0.8434864104967198, 'eval_precision': 0.8426966292134831, 'eval_recall': 0.8442776735459663, 'eval_runtime': 0.5225, 'eval_samples_per_second': 2040.184, 'eval_steps_per_second': 32.536, 'epoch': 3.0}
Full fine-tuning test results: {'eval_loss': 0.5626180171966553, 'eval_accuracy': 0.8339587242026266, 'eval_f1': 0.8368663594470046, 'eval_precision': 0.822463768115942, 'eval_recall': 0.851782363977486, 'eval_runtime': 0.5011, 'eval_samples_per_second': 2127.347, 'eval_steps_per_second': 33.926, 'epoch': 3.0}


**Analisi**: ecco il confronto finale sul test set tra i tre approcci:

| Metodo | Accuracy | F1 | Parametri allenati |
|---|---|---|---|
| SVM su feature `[CLS]` congelate | 0.796 | 0.791 | 0 (solo la SVM) |
| Fine-tuning completo | 0.834 | 0.837 | 66,955,010 (100%) |
| LoRA (`r=8` su `q_lin`/`v_lin`) | **0.843** | **0.843** | 739,586 (**1.09%**) |

Il risultato piu’ interessante non e’ solo che LoRA sia comparabile al fine-tuning completo allenando appena l'1.09% dei parametri (`print_trainable_parameters()` riporta esattamente `trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925`): LoRA lo **supera leggermente**. Non e’ un caso isolato, e i log di training spiegano il perche’:

| Epoca | Training Loss (LoRA) | Validation Loss (LoRA) |
|---|---|---|
| 1 | 0.395 | 0.376 |
| 2 | 0.323 | 0.364 |
| 3 | 0.250 | 0.376 |

A differenza del fine-tuning completo (dove la validation loss risaliva chiaramente nell'ultima epoca, vedi Esercizio 2.3), qui la validation loss resta stabile lungo tutte e tre le epoche. Con solo ~740k parametri allenabili, LoRA ha molta meno capacita’ per adattarsi eccessivamente al training set: agisce di fatto come un regolarizzatore implicito, il che su un dataset piccolo come Rotten Tomatoes (8530 esempi di training) fa la differenza tra iniziare a overfittare o no.

Come effetto collaterale, LoRA e’ stato anche piu’ veloce a parita’ di hardware: `train_runtime` di 48.3s contro 86.2s del fine-tuning completo (530.0 vs 296.9 esempi/secondo), pur usando la stessa identica architettura sotto: meno parametri allenabili significano meno gradienti da calcolare e accumulare durante la backward pass.

In generale il vantaggio di LoRA non e’ quindi solo "quasi la stessa accuratezza con molti meno parametri" (il risultato tipicamente atteso), ma qui, su un dataset di queste dimensioni, diventa anche un vantaggio netto in accuratezza grazie alla minore tendenza a overfittare, oltre che un vantaggio in tempo di training. Un'alternativa altrettanto valida (senza PEFT) sarebbe stata il *layer freezing*: congelare i primi N layer del transformer e allenare solo gli ultimi 1-2 layer piu’ la testa di classificazione, con un effetto regolarizzante simile ma piu’ grezzo (nessun adattamento, per quanto a basso rango, sui layer congelati).

#### Extra: Efficiency Summary e Conclusioni

Oltre al confronto di accuratezza gia’ visto sopra, vale la pena quantificare esplicitamente **quanto** LoRA riduca il costo computazionale, non solo che lo riduca. Calcoliamo questi numeri direttamente dalle variabili gia’ in memoria (non li scriviamo a mano), cosi’ restano corretti anche se si rilancia il notebook con iperparametri diversi.

In [14]:
lora_trainable, lora_total = peft_model.get_nb_trainable_parameters()

reduction_factor = n_trainable / lora_trainable

# Stima approssimata del risparmio di memoria per gli stati dell ottimizzatore:
# Adam mantiene 2 momenti (m, v) per ogni parametro allenabile, quindi la memoria
# per l ottimizzatore scala linearmente con il numero di parametri allenabili.
memory_savings_pct = (1 - lora_trainable / n_trainable) * 100

print(f"Full fine-tuning - parametri allenabili: {n_trainable:,}")
print(f"LoRA             - parametri allenabili: {lora_trainable:,} su {lora_total:,} totali ({100 * lora_trainable / lora_total:.2f}%)")
print(f"Fattore di riduzione dei parametri allenabili: {reduction_factor:.1f}x")
print(f"Risparmio stimato di memoria per gli stati dell ottimizzatore: {memory_savings_pct:.1f}%")

svm_acc = accuracy_score(y_test, test_pred)
svm_f1 = f1_score(y_test, test_pred)

print()
print("Approccio             | Test Accuracy | Test F1  | Parametri allenabili")
print("-" * 75)
print(f"{'Baseline (SVM)':22s}| {svm_acc:.4f}        | {svm_f1:.4f}   | 0 (DistilBERT congelato)")
print(f"{'Full fine-tuning':22s}| {test_results['eval_accuracy']:.4f}        | {test_results['eval_f1']:.4f}   | {n_trainable:,} (100%)")
print(f"{'LoRA fine-tuning':22s}| {lora_test_results['eval_accuracy']:.4f}        | {lora_test_results['eval_f1']:.4f}   | {lora_trainable:,} ({100 * lora_trainable / lora_total:.2f}%)")


Full fine-tuning - parametri allenabili: 66,955,010
LoRA             - parametri allenabili: 739,586 su 67,694,596 totali (1.09%)
Fattore di riduzione dei parametri allenabili: 90.5x
Risparmio stimato di memoria per gli stati dell ottimizzatore: 98.9%

Approccio             | Test Accuracy | Test F1  | Parametri allenabili
---------------------------------------------------------------------------
Baseline (SVM)        | 0.7964        | 0.7911   | 0 (DistilBERT congelato)
Full fine-tuning      | 0.8340        | 0.8369   | 66,955,010 (100%)
LoRA fine-tuning      | 0.8433        | 0.8435   | 739,586 (1.09%)


**Conclusioni (Esercizio 3.1)**:

- **LoRA non e’ solo "quasi altrettanto buono" del full fine-tuning: qui lo supera** (0.843 vs 0.834 di accuracy sul test), allenando appena l'1.09% dei parametri totali. Non e’ il risultato "da manuale" che ci si aspetterebbe a priori (di solito il full fine-tuning vince o pareggia), ed e’ spiegabile: con un training set piccolo come Rotten Tomatoes (8530 esempi), i ~67M di parametri allenabili del full fine-tuning danno al modello la capacita’ di iniziare a overfittare (si vede dalla validation loss che risale nell'ultima epoca, Es. 2.3), mentre i ~740k parametri di LoRA agiscono da regolarizzatore implicito.
- **Riduzione dei parametri allenabili**: un fattore di riduzione a due cifre (si veda il numero calcolato sopra), a fronte di una perdita di accuratezza nulla o addirittura di un guadagno in questo caso specifico.
- **Velocita’ di training**: LoRA ha completato le 3 epoche in circa la meta’ del tempo del full fine-tuning (48.3s vs 86.2s, cioe’ quasi 2x piu’ veloce), pur usando la stessa identica architettura sotto: meno parametri allenabili significa meno gradienti da calcolare e accumulare nella backward pass.
- **Takeaway pratico**: su task e dataset di dimensioni simili a questo, LoRA non e’ solo un compromesso "un po’ meno accurato ma molto piu’ leggero" da scegliere solo se si hanno vincoli hardware stringenti; qui e’ stata la scelta strettamente migliore su tutti gli assi misurati (accuratezza, tempo, memoria). Il caso in cui ci si aspetterebbe il full fine-tuning tornare in vantaggio e’ con dataset di training molto piu’ grandi, dove il rischio di overfitting del full fine-tuning si riduce e la maggiore capacita’ del modello puo’ essere sfruttata a pieno.
- **Limite da segnalare onestamente**: questi risultati vengono da un singolo run (un solo seed), su un dataset di validazione/test piccolo (1066 esempi ciascuno). Le differenze tra full fine-tuning e LoRA (quasi un punto percentuale) sono plausibilmente dentro il rumore statistico di run-to-run; per una conclusione piu’ solida servirebbero piu’ run con seed diversi e magari un test di significativita’ statistica sulle differenze di accuratezza.